# Session 3 - LangChain: Agents & Tools

This notebook is part of the **AI Agents & Workflows: Basics** course.

In this notebook we will learn how developers can use LangChain to build AI applications step by step.

## What we will cover

- Why LangChain exists
- How to use `ChatOpenAI`
- How to create prompt templates
- How to build simple chains
- How to create tools
- How tool calling works
- How to build a simple agent

> This notebook requires an OpenAI API key configured in `.env`.


## 0. Environment Setup

Before running this notebook, make sure you have:

1. Created a `.env` file in the repository root
2. Added your OpenAI API key
3. Installed the required dependencies

Example `.env` file:

```text
OPENAI_API_KEY=your_api_key_here

# Optional, only if needed in corporate networks
HTTP_PROXY=
HTTPS_PROXY=

LANGCHAIN_API_KEY=
LANGCHAIN_TRACING_V2=false
LANGCHAIN_PROJECT=ai-agents-workflows-basics
```


In [ ]:
import os
from dotenv import load_dotenv

# The notebook is located in notebooks/
# The .env file is expected to be in the repository root
load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("OPENAI_API_KEY loaded successfully")
else:
    print("OPENAI_API_KEY is missing")


## 1. Why LangChain?

In Session 1 we called the OpenAI API directly.

That works well for simple use cases.

But real AI applications usually need:

- reusable prompts
- structured chains
- tools
- memory
- agents
- observability

LangChain helps organize these building blocks.


## 2. First LangChain Model

LangChain provides wrappers around LLM providers.

Here we use `ChatOpenAI`.


In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

response = llm.invoke("Explain LangChain in one short paragraph.")

print(response.content)


LangChain is a framework designed to facilitate the development of applications that leverage large language models (LLMs) by providing tools and components for building, managing, and integrating LLMs with various data sources and APIs. It enables developers to create complex workflows, manage prompts, and handle interactions with LLMs in a modular way, making it easier to build applications that require natural language understanding and generation capabilities.


## 3. Prompt Templates

A prompt template allows us to reuse prompts with variables.

This is better than hardcoding text every time.


In [3]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple terms for software developers."
)

formatted_prompt = prompt.invoke({
    "topic": "AI Agents"
})

print(formatted_prompt)


messages=[HumanMessage(content='Explain AI Agents in simple terms for software developers.', additional_kwargs={}, response_metadata={})]


## 4. Prompt + Model = Chain

LangChain uses a simple composition style called **LCEL**.

LCEL means **LangChain Expression Language**.

The pipe operator `|` connects components together.


In [4]:
chain = prompt | llm

response = chain.invoke({
    "topic": "tool calling"
})

print(response.content)


Tool calling, in the context of software development, refers to the process of using or invoking a specific function, method, or service provided by a software tool or library. Here’s a simple breakdown:

1. **Tools and Libraries**: In software development, tools and libraries are collections of pre-written code that help developers perform common tasks without having to write everything from scratch. Examples include libraries for data manipulation, web frameworks, or APIs for accessing external services.

2. **Calling a Tool**: When a developer "calls" a tool, they are essentially telling their program to use a specific function or feature from that tool. This is done by writing code that references the tool's functions.

3. **How It Works**:
   - **Importing**: First, the developer needs to include or import the tool or library into their code. This makes the functions available for use.
   - **Using Functions**: The developer then calls a specific function from the tool by writing 

## 5. Add an Output Parser

By default, the model returns an AI message object.

Often we want only the text.

For that we can add an output parser.


In [5]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = prompt | llm | parser

result = chain.invoke({
    "topic": "LangChain chains"
})

print(result)


LangChain is a framework designed to help developers build applications that utilize language models (like GPT-3 or similar) in a structured and efficient way. At its core, LangChain introduces the concept of "chains," which are sequences of operations that process input data and produce output.

### What are LangChain Chains?

1. **Basic Concept**: Think of a chain as a series of steps that take an input, perform some processing, and produce an output. Each step in the chain can involve different operations, such as calling a language model, transforming data, or making decisions based on the output of previous steps.

2. **Modularity**: Chains are modular, meaning you can easily add, remove, or modify steps without having to rewrite the entire process. This makes it easier to experiment with different workflows and improve your application.

3. **Types of Chains**:
   - **Sequential Chains**: These chains execute steps one after the other. The output of one step becomes the input for

## 6. What is a Tool?

A tool is a function that an agent can use.

Examples:

- call an API
- query a database
- search documentation
- read a file
- execute business logic

In LangChain, Python functions can be converted into tools.


In [6]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Return fake weather information for a given city."""
    return f"The weather in {city} is sunny and 25°C."

print(get_weather.name)
print(get_weather.description)
print(get_weather.invoke("Sofia"))


get_weather
Return fake weather information for a given city.
The weather in Sofia is sunny and 25°C.


## 7. Another Tool

Let's create one more simple tool.

This tool simulates a knowledge base lookup.


In [8]:
@tool
def search_course_docs(query: str) -> str:
    """Search the course documentation for a given topic."""
    docs = {
        "agent": "An agent is an LLM-powered system that can use tools and make decisions.",
        "workflow": "A workflow is a predefined sequence of steps used to complete a task.",
        "rag": "RAG means Retrieval Augmented Generation. It gives the LLM external knowledge before answering.",
        "langchain": "LangChain is a framework for building LLM-powered applications."
    }

    query_lower = query.lower()

    for keyword, answer in docs.items():
        if keyword in query_lower:
            return answer

    return "No relevant course documentation found."

print(search_course_docs.invoke("What is an rag?"))


RAG means Retrieval Augmented Generation. It gives the LLM external knowledge before answering.


## 8. Manual Tool Calling

Before using a real agent, let's understand the concept manually.

The application can decide which tool to call based on the user request.

This is not yet a real LLM-powered agent.


In [10]:
def manual_tool_router(user_request: str) -> str:
    request = user_request.lower()

    if "weather" in request:
        return get_weather.invoke("Sofia")
    elif "agent" in request or "workflow" in request or "rag" in request or "langchain" in request:
        return search_course_docs.invoke(user_request)
    else:
        return "No tool selected."

print(manual_tool_router("What is an agent?"))
print(manual_tool_router("What is the weather today?"))


No tool selected.


## 9. Why Agents Are Different

The manual router above is still rule-based.

A real agent uses the LLM to decide:

- whether a tool is needed
- which tool to call
- what input to pass to the tool
- how to use the tool result

This is the key difference.


## 10. Bind Tools to the Model

Now we provide tools to the LLM.

The model can decide to request a tool call.


In [11]:
tools = [get_weather, search_course_docs]

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke(
    "What is the weather in Sofia?"
)

print(response)
print()
print("Tool calls:")
print(response.tool_calls)


content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 77, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_7396530e6a', 'id': 'chatcmpl-DtuB7fVXEELVrZl5VaYnaXM5Caqd7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ef45f-83ad-7961-bd5e-f82c30328ee3-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'Sofia'}, 'id': 'call_9R1DXatc3YufADKODnDanPjF', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 77, 'output_tokens': 15, 'total_tokens': 92, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

Tool calls:
[{'name': 'get_weat

## 11. Execute the Tool Call

The model can request a tool call, but the application must execute it.

This is very important:

> The LLM does not execute tools directly.  
> The application executes the tool call.


In [12]:
available_tools = {
    "get_weather": get_weather,
    "search_course_docs": search_course_docs,
}

for tool_call in response.tool_calls:
    selected_tool = available_tools[tool_call["name"]]
    tool_result = selected_tool.invoke(tool_call["args"])

    print("Tool name:", tool_call["name"])
    print("Tool args:", tool_call["args"])
    print("Tool result:", tool_result)


Tool name: get_weather
Tool args: {'city': 'Sofia'}
Tool result: The weather in Sofia is sunny and 25°C.


## 12. Create a Simple Agent

Now we will create a simple agent using LangChain.

This agent can:

- receive a user request
- decide if a tool is needed
- call the tool
- return a final answer


In [14]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant for the AI Agents & Workflows: Basics course. Use tools when needed."
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the dog name?"
        }
    ]
})

print(result["messages"][-1].content)


Could you please provide more context or specify which dog you are referring to? There are many dogs with various names!


## 13. Test the Agent with Course Documentation

Now let's ask a question that should use the course documentation tool.


In [15]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is RAG?"
        }
    ]
})

print(result["messages"][-1].content)


RAG stands for Retrieval Augmented Generation. It is a technique that provides a language model (LLM) with external knowledge before it generates an answer. This approach enhances the model's ability to produce more accurate and contextually relevant responses by incorporating information retrieved from external sources.


## 14. Test the Agent Without a Tool

The agent should not use tools for every question.

Sometimes the LLM can answer directly.


In [16]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Explain in one sentence why tools are useful for AI agents."
        }
    ]
})

print(result["messages"][-1].content)


Tools are useful for AI agents because they extend the agents' capabilities, allowing them to perform specific tasks, access external information, and interact with the environment more effectively.


## 15. Developer Discussion

Questions for the group:

1. What is the difference between a normal function and a LangChain tool?
2. Who decides which tool should be called?
3. Who actually executes the tool?
4. Why is tool description important?
5. What could go wrong with tool calling?

Important point:

> The quality of the tool description strongly influences whether the agent uses the tool correctly.


## 16. Exercise

Create your own tool.

Ideas:

- `get_build_status(job_name)`
- `search_documentation(query)`
- `create_ticket(title)`
- `calculate_sum(numbers)`
- `get_team_info(team_name)`

Then add it to the `tools` list and ask the agent to use it.


In [17]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant for the AI Agents & Workflows: Basics course. Use tools when needed."
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the weather in Sofia?"
        }
    ]
})

print(result["messages"][-1].content)


The weather in Sofia is sunny with a temperature of 25°C.


## 17. Key Takeaways

- LangChain helps developers structure LLM applications
- `ChatOpenAI` is a model wrapper
- Prompt templates make prompts reusable
- Chains connect components together
- Tools give agents external capabilities
- Agents decide when and how to use tools
- The application executes the selected tool
- This is the foundation for real AI Agents

Next session:

## Memory & Prompt Engineering
